# Sorting harmony tiff output into reasonable folder

In [1]:
import os
import re
import shutil
from collections import defaultdict
import tkinter as tk
from tkinter import filedialog, messagebox

class FileSorterApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("File Sorter")
        self.geometry("640x520")
        self.resizable(True, True)

        # State
        self.source_dir = tk.StringVar(value="")
        self.dest_dir = tk.StringVar(value="")
        self.file_map = defaultdict(list)  # keys: (r##, c##, f##, ch#)
        self.row_set = set()  # r##
        self.col_set = set()  # c##
        self.row_vars = {}    # display 'A','B',... -> tk.BooleanVar
        self.col_vars = {}    # display '1','2',... -> tk.BooleanVar

        self._build_ui()

    # ---------- UI LAYOUT ----------
    def _build_ui(self):
        pad = 8

        # Title
        tk.Label(self, text="TIFF File Sorter", font=("Segoe UI", 14, "bold")).pack(pady=(12, 6))

        # Folder selection
        folders = tk.Frame(self)
        folders.pack(fill="x", padx=pad, pady=(0, 8))

        # Source
        src_frame = tk.LabelFrame(folders, text="Source Folder", padx=pad, pady=pad)
        src_frame.pack(fill="x", padx=pad, pady=(0, pad))
        tk.Button(src_frame, text="Browse…", command=self._browse_source).pack(side="left")
        tk.Label(src_frame, textvariable=self.source_dir, anchor="w").pack(side="left", fill="x", expand=True, padx=(8, 0))

        # Destination
        dst_frame = tk.LabelFrame(folders, text="Destination Folder", padx=pad, pady=pad)
        dst_frame.pack(fill="x", padx=pad, pady=(0, pad))
        tk.Button(dst_frame, text="Browse…", command=self._browse_dest).pack(side="left")
        tk.Label(dst_frame, textvariable=self.dest_dir, anchor="w").pack(side="left", fill="x", expand=True, padx=(8, 0))

        # Scan / status
        scan_bar = tk.Frame(self)
        scan_bar.pack(fill="x", padx=pad, pady=(0, 8))
        self.scan_btn = tk.Button(scan_bar, text="Scan Source", state="disabled", command=self._scan_source)
        self.scan_btn.pack(side="left")
        self.status_lbl = tk.Label(scan_bar, text="Select source and destination, then click Scan.")
        self.status_lbl.pack(side="left", padx=(10, 0))

        # Selection area
        self.sel_frame = tk.Frame(self)
        self.sel_frame.pack(fill="both", expand=True, padx=pad, pady=(0, 8))

        # Rows frame
        self.rows_frame = tk.LabelFrame(self.sel_frame, text="Rows (A, B, C, …)", padx=pad, pady=pad)
        self.rows_frame.grid(row=0, column=0, sticky="nsew", padx=(0, pad), pady=(0, pad))

        # Columns frame
        self.cols_frame = tk.LabelFrame(self.sel_frame, text="Columns (1, 2, 3, …)", padx=pad, pady=pad)
        self.cols_frame.grid(row=0, column=1, sticky="nsew", padx=(pad, 0), pady=(0, pad))

        # Make selection area responsive
        self.sel_frame.columnconfigure(0, weight=1)
        self.sel_frame.columnconfigure(1, weight=1)
        self.sel_frame.rowconfigure(0, weight=1)

        # Containers for checkboxes
        self.rows_checks = tk.Frame(self.rows_frame)
        self.rows_checks.pack(fill="both", expand=True)
        self.cols_checks = tk.Frame(self.cols_frame)
        self.cols_checks.pack(fill="both", expand=True)

        # Row select/clear buttons
        rows_btns = tk.Frame(self.rows_frame)
        rows_btns.pack(fill="x", pady=(6, 0))
        tk.Button(rows_btns, text="Select All", command=self._select_all_rows).pack(side="left")
        tk.Button(rows_btns, text="Clear", command=self._clear_all_rows).pack(side="left", padx=(6, 0))

        # Column select/clear buttons
        cols_btns = tk.Frame(self.cols_frame)
        cols_btns.pack(fill="x", pady=(6, 0))
        tk.Button(cols_btns, text="Select All", command=self._select_all_cols).pack(side="left")
        tk.Button(cols_btns, text="Clear", command=self._clear_all_cols).pack(side="left", padx=(6, 0))

        # Run button
        run_bar = tk.Frame(self)
        run_bar.pack(fill="x", padx=pad, pady=(0, 14))
        self.run_btn = tk.Button(run_bar, text="Run Sorting (Move files)", state="disabled", command=self._run_sorting)
        self.run_btn.pack(side="right")

        # Footer note
        tk.Label(self, text="Tip: Leave rows/columns unselected to include ALL detected.", fg="#555").pack(pady=(0, 10))

    # ---------- Folder Browsers ----------
    def _browse_source(self):
        path = filedialog.askdirectory(parent=self, title="Select Source Folder")
        if path:
            self.source_dir.set(path)
        self._update_scan_btn_state()

    def _browse_dest(self):
        path = filedialog.askdirectory(parent=self, title="Select Destination Folder")
        if path:
            self.dest_dir.set(path)
        self._update_scan_btn_state()

    def _update_scan_btn_state(self):
        enabled = bool(self.source_dir.get()) and bool(self.dest_dir.get())
        self.scan_btn.config(state="normal" if enabled else "disabled")
        if enabled:
            self.status_lbl.config(text="Ready to scan the source folder.")
        else:
            self.status_lbl.config(text="Select source and destination, then click Scan.")

    # ---------- Scanning & UI population ----------
    def _scan_source(self):
        src = self.source_dir.get()
        dst = self.dest_dir.get()
        if not src or not os.path.isdir(src):
            messagebox.showerror("Error", "Please select a valid source folder.")
            return
        if not dst:
            messagebox.showerror("Error", "Please select a destination folder.")
            return
        os.makedirs(dst, exist_ok=True)

        # Reset previous state
        self.file_map.clear()
        self.row_set.clear()
        self.col_set.clear()
        self.row_vars.clear()
        self.col_vars.clear()
        for child in self.rows_checks.winfo_children():
            child.destroy()
        for child in self.cols_checks.winfo_children():
            child.destroy()

        pattern = re.compile(r"(r\d{2})(c\d{2})(f\d{2}).*?(ch\d{1,2})", re.IGNORECASE)

        # Scan files
        count = 0
        try:
            for name in os.listdir(src):
                if name.lower().endswith((".tiff", ".tif")):
                    m = pattern.search(name)
                    if m:
                        row, col, field, channel = m.groups()
                        row = row.lower()
                        col = col.lower()
                        field = field.lower()
                        channel = channel.lower()
                        self.row_set.add(row)
                        self.col_set.add(col)
                        self.file_map[(row, col, field, channel)].append(name)
                        count += 1
        except Exception as e:
            messagebox.showerror("Error", f"Failed to scan source folder:\n{e}")
            return

        if not self.file_map:
            self.status_lbl.config(text="No matching TIFF files found.")
            self.run_btn.config(state="disabled")
            return

        # Build display labels and checkboxes
        row_labels = sorted({self._row_code_to_letter(r) for r in self.row_set})
        col_labels = sorted({self._col_code_to_number(c) for c in self.col_set}, key=lambda x: int(x))

        # Rows
        for i, lbl in enumerate(row_labels):
            var = tk.BooleanVar(value=False)
            self.row_vars[lbl] = var
            tk.Checkbutton(self.rows_checks, text=lbl, variable=var, anchor="w").grid(row=i, column=0, sticky="w", padx=4, pady=2)

        # Columns
        for i, lbl in enumerate(col_labels):
            var = tk.BooleanVar(value=False)
            self.col_vars[lbl] = var
            tk.Checkbutton(self.cols_checks, text=lbl, variable=var, anchor="w").grid(row=i, column=0, sticky="w", padx=4, pady=2)

        self.status_lbl.config(text=f"Scanned {count} TIFF files. Select rows/columns or leave empty to include ALL.")
        self.run_btn.config(state="normal")

    # ---------- Sorting ----------
    def _run_sorting(self):
        if not self.file_map:
            messagebox.showerror("Error", "Nothing to sort. Please scan first.")
            return

        src = self.source_dir.get()
        dst = self.dest_dir.get()
        if not src or not os.path.isdir(src):
            messagebox.showerror("Error", "Source folder is not valid anymore.")
            return
        os.makedirs(dst, exist_ok=True)

        # Map selections back to r##/c## codes.
        selected_row_labels = {lbl for lbl, var in self.row_vars.items() if var.get()}
        selected_col_labels = {lbl for lbl, var in self.col_vars.items() if var.get()}

        selected_rows = set(self.row_set) if not selected_row_labels else {self._letter_to_row_code(lbl) for lbl in selected_row_labels}
        selected_cols = set(self.col_set) if not selected_col_labels else {self._number_to_col_code(lbl) for lbl in selected_col_labels}

        # Do the moves
        moved = 0
        try:
            for (row, col, field, channel), files in self.file_map.items():
                if row in selected_rows and col in selected_cols:
                    row_num = int(row[1:])
                    row_letter = chr(65 + row_num - 1)
                    col_num = int(col[1:])
                    col_folder = f"{row_letter}{col_num}"

                    target_path = os.path.join(dst, row_letter, col_folder, field, channel)
                    os.makedirs(target_path, exist_ok=True)

                    for filename in files:
                        src_path = os.path.join(src, filename)
                        dst_path = os.path.join(target_path, filename)
                        # Only move if source still exists (in case of previous runs)
                        if os.path.exists(src_path):
                            shutil.move(src_path, dst_path)
                            moved += 1
        except Exception as e:
            messagebox.showerror("Error", f"An error occurred during sorting:\n{e}")
            return

        messagebox.showinfo("Done", f"✅ Sorting complete!\nMoved files: {moved}")
        # Close the application window after completion
        try:
            self.destroy()
        except:
            pass

    # ---------- Select/Clear helpers (ADDED) ----------
    def _select_all_rows(self):
        """Check all row checkboxes (if any)."""
        for var in self.row_vars.values():
            var.set(True)

    def _clear_all_rows(self):
        """Uncheck all row checkboxes (if any)."""
        for var in self.row_vars.values():
            var.set(False)

    def _select_all_cols(self):
        """Check all column checkboxes (if any)."""
        for var in self.col_vars.values():
            var.set(True)

    def _clear_all_cols(self):
        """Uncheck all column checkboxes (if any)."""
        for var in self.col_vars.values():
            var.set(False)

    # ---------- Helpers ----------
    @staticmethod
    def _row_code_to_letter(r_code: str) -> str:
        # r01 -> A
        r_num = int(r_code[1:])
        return chr(65 + r_num - 1)

    @staticmethod
    def _letter_to_row_code(letter: str) -> str:
        # A -> r01
        r_num = ord(letter.upper()) - 65 + 1
        return f"r{r_num:02d}"

    @staticmethod
    def _col_code_to_number(c_code: str) -> str:
        # c01 -> "1"
        return str(int(c_code[1:]))

    @staticmethod
    def _number_to_col_code(num_str: str) -> str:
        # "1" -> c01
        return f"c{int(num_str):02d}"

# Run the app
if __name__ == "__main__":
    app = FileSorterApp()
    app.mainloop()